## Notebook 4 – Fine-tuning Wav2Vec2 for Speech Emotion Recognition

• Load processed dataset

• Prepare train/validation/test splits

• Load pretrained Wav2Vec2 model

• Freeze feature extractor

• Fine-tune the classification head

• Evaluate validation performance

• Save the trained model and artifacts

## 1. Setup and imports

In [1]:
import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = '/Users/devanshbansal/Desktop/ser'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('NumPy:', np.__version__)

NumPy: 1.26.4


## 2. Load the extracted features

In [2]:
# ============================================================
# Load Processed Dataset from Notebook 3
# ============================================================

from pathlib import Path
import pickle

BASE_DIR = Path.cwd()

# Go back to SER project root
PROJECT_ROOT = BASE_DIR.parent.parent.parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

DATASET_PATH = OUTPUT_DIR / "processed_audio_dataset.pkl"

print("Loading processed dataset from:")
print(DATASET_PATH)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"\nProcessed dataset not found:\n{DATASET_PATH}\n"
        "Run Notebook 3 first."
    )

with open(DATASET_PATH, "rb") as f:
    processed_df = pickle.load(f)

print("=" * 70)
print("Processed Dataset Loaded Successfully")
print("=" * 70)

print(f"Rows               : {len(processed_df)}")
print(f"Columns            : {len(processed_df.columns)}")
print(f"Emotion Classes    : {processed_df['emotion'].nunique()}")
print(f"Sample Rate(s)     : {processed_df['sample_rate'].unique()}")

display(processed_df.head())

Loading processed dataset from:
/Users/devanshbansal/Desktop/ser/outputs/processed_audio_dataset.pkl
Processed Dataset Loaded Successfully
Rows               : 1440
Columns            : 14
Emotion Classes    : 7
Sample Rate(s)     : [16000]


,file_path,filename,emotion,emotion_code,actor,gender,intensity,statement,statement_text,repetition,vocal_channel,modality,waveform,sample_rate
0,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-01-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,1,speech,audio_only,"[3.509954e-06, -4.807788e-06, 6.262959e-06, -7...",16000
1,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-01-02-01.wav,calm,01,1,male,normal,1,Kids are talking by the door,2,speech,audio_only,"[-8.7073524e-05, -0.00017682035, 4.0303014e-05...",16000
2,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-01-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,1,speech,audio_only,"[0.00024561264, 0.00046495523, 0.00055057195, ...",16000
3,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-01-01-02-02-01.wav,calm,01,1,male,normal,2,Dogs are sitting by the door,2,speech,audio_only,"[0.00038872915, 0.000337521, -1.9962981e-05, -...",16000
4,/Users/devanshbansal/Desktop/ser/dataset/Audio...,03-01-02-01-01-01-01.wav,calm,02,1,male,normal,1,Kids are talking by the door,1,speech,audio_only,"[0.0003747411, 1.66967e-05, -1.6409602e-05, 1....",16000


## 3. Load LabelEncoder


In [3]:
# ==========================================================
# Prepare Waveforms and Labels
# ==========================================================

from sklearn.preprocessing import LabelEncoder

# Keep only successfully processed samples
processed_df = processed_df.dropna(subset=["waveform"]).reset_index(drop=True)

# Audio waveforms
X = processed_df["waveform"].tolist()

# Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(processed_df["emotion"])

print("=" * 70)
print("Dataset Prepared")
print("=" * 70)

print(f"Total Samples   : {len(X)}")
print(f"Emotion Classes : {len(label_encoder.classes_)}")
print(f"Classes         : {list(label_encoder.classes_)}")

print("\nFirst waveform length :", len(X[0]), "samples")
print("First label            :", y[0])

print("=" * 70)
print(processed_df["emotion"].unique())

assert processed_df["emotion"].nunique() == 7

Dataset Prepared
Total Samples   : 1440
Emotion Classes : 7
Classes         : ['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']

First waveform length : 52853 samples
First label            : 1
['calm' 'happy' 'sad' 'angry' 'fearful' 'disgust' 'surprised']


## 4. Train / validation / test split

In [4]:
# ==========================================================
# Train / Validation / Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

indices = np.arange(len(processed_df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=42,
    stratify=y
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.125,      # 0.125 × 0.80 = 0.10
    random_state=42,
    stratify=y[train_idx]
)

train_df = processed_df.iloc[train_idx].reset_index(drop=True)
val_df   = processed_df.iloc[val_idx].reset_index(drop=True)
test_df  = processed_df.iloc[test_idx].reset_index(drop=True)

print("=" * 70)
print("Dataset Split Completed")
print("=" * 70)

print(f"Training Samples   : {len(train_df)}")
print(f"Validation Samples : {len(val_df)}")
print(f"Testing Samples    : {len(test_df)}")

print("\nEmotion Distribution")

print("\nTrain")
print(train_df["emotion"].value_counts().sort_index())

print("\nValidation")
print(val_df["emotion"].value_counts().sort_index())

print("\nTest")
print(test_df["emotion"].value_counts().sort_index())

print("=" * 70)

Dataset Split Completed
Training Samples   : 1134
Validation Samples : 162
Testing Samples    : 144

Emotion Distribution

Train
emotion
angry        151
calm         227
disgust      152
fearful      151
happy        151
sad          151
surprised    151
Name: count, dtype: int64

Validation
emotion
angry        22
calm         32
disgust      21
fearful      22
happy        21
sad          22
surprised    22
Name: count, dtype: int64

Test
emotion
angry        19
calm         29
disgust      19
fearful      19
happy        20
sad          19
surprised    19
Name: count, dtype: int64


## 5.Convert to Hugging Face Dataset


In [5]:
# ==========================================================
# Convert to Hugging Face Dataset
# ==========================================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["waveform", "emotion"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["waveform", "emotion"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["waveform", "emotion"]]
)

print("=" * 70)
print("Hugging Face Datasets Created")
print("=" * 70)

print(f"Train      : {len(train_dataset)}")
print(f"Validation : {len(val_dataset)}")
print(f"Test        : {len(test_dataset)}")

print("\nExample:")
print(train_dataset[0])

print("=" * 70)
print(train_df["emotion"].value_counts())
print(val_df["emotion"].value_counts())
print(test_df["emotion"].value_counts())

Hugging Face Datasets Created
Train      : 1134
Validation : 162
Test        : 144

Example:
{'waveform': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

## 6. Load Feature Extractor

In [6]:
# ==========================================================
# Load Wav2Vec2 Processor
# ==========================================================

from transformers import AutoProcessor

MODEL_NAME = "facebook/wav2vec2-base"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print("=" * 70)
print("Processor Loaded Successfully")
print("=" * 70)
print("Model :", MODEL_NAME)
print("Target Sample Rate :", processor.feature_extractor.sampling_rate)
print("=" * 70)

Processor Loaded Successfully
Model : facebook/wav2vec2-base
Target Sample Rate : 16000


## 7. Verify Feature Extractor

In [7]:
# ==========================================================
# Verify Hugging Face Datasets
# ==========================================================

print("=" * 70)
print("Dataset Verification")
print("=" * 70)

print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")

print("\nColumns:")
print(train_dataset.column_names)

print("\nExample sample:")
example = train_dataset[0]

print(f"Emotion : {example['emotion']}")
print(f"Waveform Length : {len(example['waveform'])} samples")
print(f"Waveform dtype  : {type(example['waveform'])}")

print("\nWaveform Statistics")
print(f"Min  : {np.min(example['waveform']):.4f}")
print(f"Max  : {np.max(example['waveform']):.4f}")
print(f"Mean : {np.mean(example['waveform']):.4f}")
print(f"Std  : {np.std(example['waveform']):.4f}")

print("=" * 70)
print("Verification Passed ✓")
print("=" * 70)

Dataset Verification
Train samples      : 1134
Validation samples : 162
Test samples       : 144

Columns:
['waveform', 'emotion']

Example sample:
Emotion : disgust
Waveform Length : 65666 samples
Waveform dtype  : <class 'list'>

Waveform Statistics
Min  : -0.9925
Max  : 1.0000
Mean : -0.0000
Std  : 0.0754
Verification Passed ✓


## 8.  Tokenize Audio

In [8]:
# ==========================================================
# Load Wav2Vec2 Feature Extractor / Processor
# ==========================================================

from transformers import AutoFeatureExtractor

MODEL_NAME = "facebook/wav2vec2-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_NAME
)

print("=" * 70)
print("Feature Extractor Loaded")
print("=" * 70)

print("Model:")
print(MODEL_NAME)

print("\nSampling Rate:")
print(feature_extractor.sampling_rate)

print("\nPadding:")
print(feature_extractor.padding_value)

print("=" * 70)

Feature Extractor Loaded
Model:
facebook/wav2vec2-base

Sampling Rate:
16000

Padding:
0.0


## 9. Apply Tokenization


In [9]:
# ==========================================================
# Tokenize Audio
# ==========================================================

def preprocess(batch):

    audio = batch["waveform"]

    features = feature_extractor(
        audio,
        sampling_rate=16000,
        return_attention_mask=True
    )

    return {
        "input_values": features["input_values"][0],
        "attention_mask": features["attention_mask"][0],
        "label": label_encoder.transform([batch["emotion"]])[0]
    }


train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

print("=" * 70)
print("Audio Tokenization Completed")
print("=" * 70)

print(train_dataset)

print("\nExample Keys:")
print(train_dataset[0].keys())

python(74634) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Map: 100%|██████████| 144/144 [00:03<00:00, 46.94 examples/s]

Audio Tokenization Completed
Dataset({
    features: ['waveform', 'emotion', 'input_values', 'attention_mask', 'label'],
    num_rows: 1134
})

Example Keys:
dict_keys(['waveform', 'emotion', 'input_values', 'attention_mask', 'label'])


In [10]:
train_dataset = train_dataset.remove_columns(
    ["waveform","emotion"]
)

val_dataset = val_dataset.remove_columns(
    ["waveform","emotion"]
)

test_dataset = test_dataset.remove_columns(
    ["waveform","emotion"]
)

## 10. Create Data Collator

In [11]:
# ============================================================
# Create Data Collator
# ============================================================
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=processor.feature_extractor,
    padding=True,
    return_tensors="pt"
)

print("=" * 70)
print("Data Collator Created")
print("=" * 70)

print(data_collator)

Data Collator Created
DataCollatorWithPadding(tokenizer=Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}
, padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')


## 11. Load Wav2Vec2 Model

In [12]:
from transformers import Wav2Vec2ForSequenceClassification


NUM_LABELS = len(label_encoder.classes_)


model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=NUM_LABELS,
    label2id={
        label:i
        for i,label in enumerate(label_encoder.classes_)
    },
    id2label={
        i:label
        for i,label in enumerate(label_encoder.classes_)
    },
    ignore_mismatched_sizes=True
)


# Freeze CNN feature extractor
model.wav2vec2.feature_extractor._freeze_parameters()


print("="*70)
print("Wav2Vec2 Model Loaded")
print("="*70)
print(model.config.id2label)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 36836.55it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 Model Loaded
{0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful', 4: 'happy', 5: 'sad', 6: 'surprised'}


12. Verify Model

In [13]:
# ============================================================
# Verify Model Configuration
# ============================================================

print("=" * 70)
print("Model Configuration")
print("=" * 70)

print(f"Hidden Size          : {model.config.hidden_size}")
print(f"Number of Labels     : {model.config.num_labels}")
print(f"Vocabulary Size      : {model.config.vocab_size}")
print(f"Mask Time Prob       : {model.config.mask_time_prob}")
print(f"Final Dropout        : {model.config.final_dropout}")
print(f"Attention Dropout    : {model.config.attention_dropout}")
print(f"Hidden Dropout       : {model.config.hidden_dropout}")
print(f"LayerDrop            : {model.config.layerdrop}")

print("\nEmotion Mapping")

for idx, label in model.config.id2label.items():
    print(f"{idx} -> {label}")

print("=" * 70)

Model Configuration
Hidden Size          : 768
Number of Labels     : 7
Vocabulary Size      : 32
Mask Time Prob       : 0.05
Final Dropout        : 0.0
Attention Dropout    : 0.1
Hidden Dropout       : 0.1
LayerDrop            : 0.0

Emotion Mapping
0 -> angry
1 -> calm
2 -> disgust
3 -> fearful
4 -> happy
5 -> sad
6 -> surprised


In [14]:
print("="*70)
print("NOTEBOOK 4 COMPLETED SUCCESSFULLY")
print("="*70)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Testing samples    : {len(test_dataset)}")

print(f"Classes            : {list(label_encoder.classes_)}")

print("\nStatus:")
print("✓ Dataset Loaded")
print("✓ Labels Encoded")
print("✓ Train/Validation/Test Split")
print("✓ Hugging Face Dataset Created")
print("✓ Audio Tokenized")
print("✓ Data Collator Ready")
print("✓ Wav2Vec2 Model Ready")

print("\nNext Notebook:")
print("Notebook 5 - Fine-tune Wav2Vec2")

print("="*70)

NOTEBOOK 4 COMPLETED SUCCESSFULLY
Training samples   : 1134
Validation samples : 162
Testing samples    : 144
Classes            : ['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']

Status:
✓ Dataset Loaded
✓ Labels Encoded
✓ Train/Validation/Test Split
✓ Hugging Face Dataset Created
✓ Audio Tokenized
✓ Data Collator Ready
✓ Wav2Vec2 Model Ready

Next Notebook:
Notebook 5 - Fine-tune Wav2Vec2
